# Grammar scoring from speech — SHL Hiring Assessment 2026
Khushi (@starkhushi)

Each clip is 45–60 s of spontaneous spoken English and gets a grammar score on SHL's 1–5 MOS rubric. The metric is
RMSE, and I also report Pearson correlation.

The short version of what I ended up with: ridge/SVR regressors on frozen features — WavLM-large and the
Whisper-large-v3 encoder for the audio, RoBERTa-large and Qwen2.5 embeddings of two different transcripts for the
text — stacked with non-negative least squares, plus one LoRA fine-tune of the Whisper encoder as an extra member.
Public leaderboard 0.3485 (predicting the mean gives 1.005); speaker-grouped cross-validation about 0.51 RMSE /
0.86 Pearson on the 732 real speech clips.

Most of the gains came from looking at the data rather than from model choice:
- the 37 clips labelled 0 are synthetic noise, not speech;
- the training set repeats speakers while every test speaker is new, so random K-fold is misleading here;
- the training labels are noisy (about 0.33 RMSE of rater disagreement), which explains why CV and the leaderboard
  sit so far apart and justifies two small adjustments at the end;
- Whisper quietly fixes grammar when it transcribes, so I also use a verbatim CTC transcript.

The heavy feature extraction (WavLM, Whisper, the transcripts, the LLM embeddings, the fine-tunes) runs on GPU and
its outputs are attached as inputs; that code is in the GitHub repository,
<https://github.com/starkhushi/shl-2026-grammar-scoring>. This notebook refits every model from those features, prints
the metrics — including the RMSE on the training data — and writes `submission.csv`. It takes about 15 minutes on CPU.

**Sections:** 1. Data · 2. Looking at the data first · 3. Validation · 4. Pipeline · 5. Layer sweep ·
6. Models, stacking, train RMSE · 7. What is the model reacting to? · 8. What worked, what didn't, limitations

In [ ]:
import glob, os, hashlib, warnings, json
import numpy as np, pandas as pd, soundfile as sf
import matplotlib.pyplot as plt
from scipy.optimize import nnls
from scipy.stats import pearsonr, spearmanr
from sklearn.cluster import AgglomerativeClustering
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
SR = 16000

def find(pattern):
    # Kaggle mount paths change between images, so I locate inputs by glob
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True), key=len)
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

tr_csv = find('Dataset_Final/train.csv'); ROOT = os.path.dirname(tr_csv)
train = pd.read_csv(tr_csv); test = pd.read_csv(os.path.join(ROOT, 'test.csv'))
train['path'] = [os.path.join(ROOT, 'train', f) for f in train.filename]
test['path'] = [os.path.join(ROOT, 'test', f) for f in test.filename]
for d in (train, test):
    d['dur'] = [sf.info(p).duration for p in d.path]
print(f'train {len(train)} clips, test {len(test)} clips; durations train {train.dur.min():.0f}-{train.dur.max():.0f}s, '
      f'test {test.dur.min():.0f}-{test.dur.max():.0f}s; sample rate {sf.info(train.path[0]).samplerate} Hz mono')

## 1. Data
769 training clips with a score and 216 test clips, 16 kHz mono. Scores come in 0.5 steps. The rubric defines 1–5 but
37 training clips are labelled 0 (more on that below). `test.csv` has placeholder labels, and the provided
`sample_submission.csv` does not match `test.csv` at all (204 rows, half of them training filenames), so I key the
submission on `test.csv`.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
vc = train.label.value_counts().sort_index()
ax[0].bar(vc.index, vc.values, width=0.4, color=['#c44' if k == 0 else '#467' for k in vc.index])
ax[0].set_xlabel('grammar score (train)'); ax[0].set_ylabel('clips'); ax[0].set_title('Label distribution (red = the 37 zeros)')
ax[1].hist(train.dur, bins=30, alpha=.7, label='train'); ax[1].hist(test.dur, bins=30, alpha=.7, label='test')
ax[1].set_xlabel('clip duration (s)'); ax[1].legend(); ax[1].set_title('Durations')
plt.tight_layout(); plt.show()
nz = train.label[train.label > 0]
print(f'non-zero labels: mean {nz.mean():.3f}, std {nz.std():.3f}  (a constant prediction therefore scores ~{nz.std():.2f} RMSE)')

## 2. Looking at the data first
### The label-0 clips are noise
The 0s don't exist in the rubric, so I checked the signal. All 37 have a peak amplitude of exactly 0.95, energy in
every single frame and a spectral flatness of 0.84 — that's white noise, not a person. A two-condition rule catches all
37 with no false positives, and it fires on none of the test clips (a constant-mean submission scored 1.005, which is
what you'd get if the test set had no zeros). So I drop them from training, run CV on the remaining 732 clips, and keep
the rule only as a guard at inference.

In [ ]:
def signal_stats(p):
    x, _ = sf.read(p, dtype='float32')
    fr = x[:len(x) // 400 * 400].reshape(-1, 400)                      # 25 ms frames
    rms = np.sqrt((fr ** 2).mean(1) + 1e-12)
    S = np.abs(np.fft.rfft(fr * np.hanning(400), axis=1)) + 1e-9
    return dict(peak=float(np.abs(x).max()), voiced=float((rms > max(rms.max() * .02, 1e-4)).mean()),
                flatness=float(np.median(np.exp(np.log(S).mean(1)) / S.mean(1))))

st = pd.DataFrame([signal_stats(p) for p in train.path]); st['label'] = train.label.values
st_te = pd.DataFrame([signal_stats(p) for p in test.path])
is_noise = lambda d: (np.isclose(d.peak, 0.95, atol=1e-3)) & (d.voiced > 0.999)
train['noise'] = is_noise(st).values; test['noise'] = is_noise(st_te).values
print(st.groupby(st.label == 0)[['peak', 'voiced', 'flatness']].mean().rename(index={True: 'label 0', False: 'label > 0'}).round(3))
print(f'rule hits: train {train.noise.sum()} (all label 0: {(train.label[train.noise] == 0).all()}), test {test.noise.sum()}')

fig, ax = plt.subplots(2, 2, figsize=(11, 4.6))
for r, (name, p) in enumerate([('label-0 clip (noise)', train.path[train.label == 0].iloc[0]),
                               ('label-4 clip (speech)', train.path[train.label == 4].iloc[0])]):
    x, _ = sf.read(p, dtype='float32'); t = np.arange(len(x)) / SR
    ax[r, 0].plot(t[:SR * 10], x[:SR * 10], lw=.3); ax[r, 0].set_title(f'{name}: waveform (first 10 s)'); ax[r, 0].set_ylim(-1, 1)
    ax[r, 1].specgram(x[:SR * 10], Fs=SR, NFFT=512, noverlap=256, cmap='magma'); ax[r, 1].set_title('spectrogram')
plt.tight_layout(); plt.show()

### 212 of the 216 test filenames also exist in train
That looked like a leak, so I checked: no test file is byte-identical to its namesake, the waveforms are unrelated, and
the namesake is never the nearest neighbour in embedding space. The names are just reused.

In [ ]:
shared = [f for f in test.filename if f in set(train.filename)]
h = lambda p: hashlib.md5(open(p, 'rb').read()).hexdigest()
same = sum(h(os.path.join(ROOT, 'test', f)) == h(os.path.join(ROOT, 'train', f)) for f in shared)
print(f'{len(shared)} shared filenames; byte-identical pairs: {same}')

### The training set repeats speakers; the test set doesn't
Lower layers of a speech model mostly encode the voice. Taking WavLM-large layer 4 (mean over time) and cosine
similarity, more than half of the training clips have a near-identical twin elsewhere in train — the same person
answering a different prompt — and twins' labels are only 0.25 apart on average versus 0.44 for random pairs. Test
clips have no such twin in train. So an audio model judged with random folds gets credit for recognising voices, which
is worth nothing on new speakers. This changed which models looked good, and it's the first thing I'd tell someone
starting on this dataset.

In [ ]:
idx = pd.read_csv(os.path.join(os.path.dirname(find('emb_wavlmL_mean.npy')), 'index.csv'))   # row order of all .npy files
assert (idx.filename[idx.split == 'train'].values == train.filename.values).all() and (idx.filename[idx.split == 'test'].values == test.filename.values).all()
IS_TR = (idx.split == 'train').values; IS_TE = ~IS_TR
KEEP = IS_TR & ~np.r_[train.noise.values, test.noise.values]          # 732 real-speech training clips
y = train.label.values[~train.noise.values].astype(float)
E = {k: np.load(find(f), mmap_mode='r') for k, f in
     [('wavlmL_m', 'emb_wavlmL_mean.npy'), ('wavlmL_s', 'emb_wavlmL_std.npy'), ('whisper_m', 'emb_whisper_mean.npy'),
      ('whisper_s', 'emb_whisper_std.npy'), ('roberta', 'emb_roberta.npy'), ('qwen', 'emb_qwen.npy'),
      ('qwen7_whisper', 'emb_qwen7_whisper.npy'), ('qwen7_ctc', 'emb_qwen7_ctc.npy'), ('roberta_ctc', 'emb_roberta_ctc.npy')]}
print({k: v.shape for k, v in E.items()})

V = np.array(E['wavlmL_m'][:, 4], dtype=np.float64); V -= V[KEEP].mean(0); V /= np.linalg.norm(V, axis=1, keepdims=True)
S_tt = V[KEEP] @ V[KEEP].T; np.fill_diagonal(S_tt, -1); nn_tr = S_tt.max(1)
nn_te = (V[IS_TE] @ V[KEEP].T).max(1)
j = S_tt.argmax(1); dl = np.abs(y - y[j])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].hist(nn_tr, bins=40, alpha=.7, label='train clip -> nearest other train clip'); ax[0].hist(nn_te, bins=40, alpha=.7, label='test clip -> nearest train clip')
ax[0].set_xlabel('cosine similarity (WavLM-large layer 4)'); ax[0].legend(fontsize=8); ax[0].set_title('Voice twins exist in train, not across train/test')
bins = [0, .6, .7, .8, .9, 1.01]; cat = pd.cut(nn_tr, bins)
ax[1].bar(range(5), pd.Series(dl).groupby(cat).mean().values); ax[1].set_xticks(range(5)); ax[1].set_xticklabels([f'{a:.1f}-{b:.1f}' for a, b in zip(bins[:-1], bins[1:])])
ax[1].set_xlabel('similarity to nearest train clip'); ax[1].set_ylabel('|label difference|'); ax[1].set_title('The closer the voice, the closer the label')
plt.tight_layout(); plt.show()
print(f'train clips with a >0.9 twin: {(nn_tr > .9).mean():.0%}  | test clips with a >0.9 twin in train: {(nn_te > .9).mean():.0%}')

## 3. Validation
I cluster clips into speakers (average-linkage cosine clustering on the layer-4 voice embedding, threshold 0.30,
about 376 clusters) and build folds with `StratifiedGroupKFold` so a speaker never sits on both sides of a split:
5 folds × 3 seeds, 15 fits per model, always scored on the real labels. Every "CV" number below is from this setup.
It tracked the public leaderboard much better than random folds did — random-fold CV preferred models that then
scored worse on the board.

In [ ]:
GROUP_TH, TWIN_TH, SEEDS = 0.30, 0.20, [42, 101, 2024]
cl = lambda th, m: AgglomerativeClustering(n_clusters=None, distance_threshold=th, metric='cosine', linkage='average').fit_predict(V[m])
groups, twins_tr, twins_te = cl(GROUP_TH, KEEP), cl(TWIN_TH, KEEP), cl(TWIN_TH, IS_TE)
ybin = np.round(y * 2).astype(int)
SPLITS = [s for seed in SEEDS for s in StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(V[KEEP], ybin, groups)]
print(f'{len(set(groups))} speaker clusters (largest {pd.Series(groups).value_counts().max()} clips); {len(SPLITS)} train/val splits')

## 4. Pipeline
```
wav ─┬─ WavLM-large (frozen)            layer 20/21, mean & std over time ─┐
     ├─ Whisper-large-v3 encoder (frozen) layers 30-32                      ├─ per (model, layer): scaler → Ridge and SVR
     ├─ Whisper transcript ─┬─ RoBERTa-large  L20/21 ─────────────────────┤      (36 members, 15 fits each)
     │                      ├─ Qwen2.5-1.5B   L17/18 ─────────────────────┤
     │                      └─ Qwen2.5-7B     L16-22 ─────────────────────┤
     └─ CTC transcript ─────┴─ RoBERTa-large, Qwen2.5-7B ─────────────────┤
     + Whisper encoder LoRA fine-tune (same speaker folds) ────────────────┘
                                                                            │
          NNLS on out-of-fold predictions → linear calibration → twin smoothing → 0.8·stack + 0.2·WavLM fine-tune
```
Why frozen features and small regressors: 732 labelled clips isn't much. My first end-to-end fine-tune (WavLM-large,
7.6 GPU hours) scored 0.585 alone, worse than a ridge on frozen features, and only earns a 20 % blend weight. The
upper layers of these models already encode fluency and syntax; a ridge or SVR on 1–4k dimensions with ~600 rows is
easy to regularise and quick to validate properly.

Why two transcripts: Whisper's decoder is a language model, so it writes fluent text and silently turns "there are
there are play tools" into "there are play tools". The CTC transcript (wav2vec2, no LM) misspells a lot but keeps the
repetitions and broken sentences the rubric penalises. Qwen2.5-7B reading the CTC transcript is the best text member.

In [ ]:
wt = pd.read_csv(find('text.csv')); ct = pd.read_csv(find('text_ctc.csv'))
assert (wt.filename.values == idx.filename.values).all() and (ct.filename.values == idx.filename.values).all()
for i in [0, 3]:
    print(f'--- {idx.filename[i]}  (label {train.label[i]})\nWhisper : {wt.text[i][:170]}\nCTC     : {ct.ctc[i][:170]}\n')

## 5. Layer sweep
I swept every layer of every model with a ridge under grouped CV. The signal is in the upper-middle layers: WavLM-large
peaks at layer 21 of 25, the Whisper encoder at 30–32 of 33, RoBERTa-large around 20, Qwen2.5 around 17–18 — the
layers that carry syntax and discourse rather than acoustics (bottom) or next-token prediction (top).

In [ ]:
sweep = pd.read_csv(find('layer_sweep.csv'))
fig, ax = plt.subplots(figsize=(10, 3.6))
for enc, g in sweep.groupby('enc'):
    ax.plot(g.layer, g.rmse, marker='.', label=enc)
ax.set_xlabel('hidden layer'); ax.set_ylabel('grouped-CV RMSE (ridge)'); ax.set_ylim(.55, .95); ax.legend(ncol=3, fontsize=8)
ax.set_title('Layer sweep: where the grammar signal lives'); plt.show()
print(sweep.loc[sweep.groupby('enc').rmse.idxmin()].round(4).to_string(index=False))

## 6. Models, stacking and the training RMSE
36 members (18 model-layer blocks × ridge and SVR), each fitted 15 times on the grouped folds. Stacking weights come
from non-negative least squares on the out-of-fold predictions, and the stack is nested — weights and calibration are
fitted on four folds and scored on the fifth — so the CV number is honest. Members are fitted to speaker-mean targets
(explained in the label-noise section further down); scoring always uses the raw labels.

In [ ]:
QL = [10, 12, 14, 16, 18, 20, 22, 24, 26, 28]            # layers stored for the Qwen-7B arrays
BLOCKS = {'wavlmL_m': [21, 20], 'wavlmL_s': [21, 22], 'whisper_m': [32, 30], 'whisper_s': [32, 31],
          'roberta': [20, 21], 'qwen': [17, 18], 'qwen7_whisper': [18, 16], 'qwen7_ctc': [18, 20], 'roberta_ctc': [19, 18],
          'q7p_ctc': [20, 22], 'q7p_whisper': [14, 16]}          # q7p_* = Qwen-7B asked a grammar question, last-token state
SVR_C = {k: (10 if k.startswith(('wavlm', 'whisper')) else 1) for k in BLOCKS}
E['q7p_ctc'] = np.load(find('emb_q7p_ctc.npy'), mmap_mode='r'); E['q7p_whisper'] = np.load(find('emb_q7p_whisper.npy'), mmap_mode='r')
ALPHAS = {k: ([300, 3000, 30000] if SVR_C[k] == 1 else [30, 100, 300, 1000, 3000]) for k in BLOCKS}
col = lambda name, L: (QL.index(L) if name.startswith(('qwen7', 'q7p')) else L)

DENOISE = True                      # train on speaker-mean labels (see the label-noise section); scoring uses the raw labels
gmean = lambda v, g: pd.Series(v).groupby(g).transform('mean').values

def cv_fit(make, X, Xte):
    oof = np.zeros(len(y)); cnt = np.zeros(len(y)); pte = 0
    for trn, val in SPLITS:
        yt = gmean(y[trn], twins_tr[trn]) if DENOISE else y[trn]          # computed inside the training fold only
        sc = StandardScaler().fit(X[trn]); m = make().fit(sc.transform(X[trn]), yt)
        oof[val] += m.predict(sc.transform(X[val])); cnt[val] += 1; pte = pte + m.predict(sc.transform(Xte))
    return oof / cnt, pte / len(SPLITS)

M, T, rows = {}, {}, []
for name, layers in BLOCKS.items():
    for L in layers:
        X = np.array(E[name][KEEP, col(name, L)], dtype=np.float32); Xte = np.array(E[name][IS_TE, col(name, L)], dtype=np.float32)
        best = min(((rmse(y, o), o, p, a) for a in ALPHAS[name] for o, p in [cv_fit(lambda a=a: Ridge(alpha=a), X, Xte)]), key=lambda t: t[0])
        M[f'{name}_L{L}_ridge'], T[f'{name}_L{L}_ridge'] = best[1], best[2]
        o, p = cv_fit(lambda: SVR(C=SVR_C[name], epsilon=0.1), X, Xte)
        M[f'{name}_L{L}_svr'], T[f'{name}_L{L}_svr'] = o, p
        rows.append(dict(model=name, layer=L, ridge_rmse=best[0], ridge_alpha=best[3], svr_rmse=rmse(y, o)))
        print(f'{name:14s} L{L:2d}  ridge {best[0]:.4f} (alpha {best[3]})  svr {rmse(y, o):.4f}', flush=True)
# the LoRA fine-tune of the Whisper encoder was trained on these same speaker folds (seed 42),
# so its out-of-fold predictions can go into the stack like any other member
ft2 = pd.read_csv(find('oof_ft2.csv')).set_index('filename').loc[train.filename[~train.noise.values]]
ft2_te = pd.read_csv(find('test_ft2.csv')); assert (ft2_te.filename.values == test.filename.values).all()
M['ft_whisper_lora'], T['ft_whisper_lora'] = ft2.oof.values, ft2_te.pred.values
print(f'fine-tuned Whisper-encoder member: grouped OOF rmse {rmse(y, ft2.oof.values):.4f}')
M, T = pd.DataFrame(M), pd.DataFrame(T)
members = pd.DataFrame(rows).round(4); members

In [ ]:
SMOOTH = 0.5

def stack(cols, smooth=SMOOTH):
    Mv, Tv = M[cols].values, T[cols].values
    nest = np.zeros(len(y))
    for trn, val in SPLITS[:5]:                                   # nested: weights + calibration never see the scored fold
        w_, _ = nnls(Mv[trn], y[trn]); w_ /= w_.sum(); a_, b_ = np.polyfit(Mv[trn] @ w_, y[trn], 1)
        p = a_ * (Mv[val] @ w_) + b_
        nest[val] = np.clip((1 - smooth) * p + smooth * gmean(p, twins_tr[val]), 1, 5)
    w, _ = nnls(Mv, y); w /= w.sum(); a, b = np.polyfit(Mv @ w, y, 1)
    fit = np.clip(a * (Mv @ w) + b, 1, 5)                         # in-sample ("training data") prediction
    pte = a * (Tv @ w) + b
    pte = np.clip((1 - smooth) * pte + smooth * gmean(pte, twins_te), 1, 5)
    return dict(cv_rmse=rmse(y, nest), cv_pearson=pearsonr(y, nest)[0], train_rmse=rmse(y, fit), train_pearson=pearsonr(y, fit)[0],
                nest=nest, fit=fit, test=pte, w=pd.Series(w, cols), a=a, b=b)

audio = [c for c in M if c.split('_L')[0] in ('wavlmL_m', 'wavlmL_s', 'whisper_m', 'whisper_s')]
text = [c for c in M if c not in audio and c != 'ft_whisper_lora']
frozen = audio + text
abl = {'audio only (frozen)': stack(audio), 'text only': stack(text), 'frozen audio + text, no twin smoothing': stack(frozen, 0.0),
       'frozen audio + text': stack(frozen), 'frozen + fine-tuned member (final)': stack(list(M))}
print(pd.DataFrame({k: {m: v[m] for m in ('cv_rmse', 'cv_pearson', 'train_rmse', 'train_pearson')} for k, v in abl.items()}).T.round(4))
FINAL = abl['frozen + fine-tuned member (final)']
print(f"\n==> RMSE on the TRAINING DATA (in-sample fit of the final stack): {FINAL['train_rmse']:.4f}   | Pearson {FINAL['train_pearson']:.4f}")
print(f"==> Speaker-grouped cross-validated RMSE (honest estimate):      {FINAL['cv_rmse']:.4f}   | Pearson {FINAL['cv_pearson']:.4f}")
print(f"==> constant-mean baseline RMSE: {rmse(y, np.full_like(y, y.mean())):.4f}; calibration a={FINAL['a']:.3f}, b={FINAL['b']:.3f}")

In [ ]:
w = FINAL['w'][FINAL['w'] > 0].sort_values()
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].barh(w.index, w.values, color=['#467' if any(k in i for k in ('wavlm', 'whisper')) else '#c84' for i in w.index])
ax[0].set_title('NNLS stacking weights (blue = audio, orange = text)'); ax[0].tick_params(axis='y', labelsize=7)
ax[1].scatter(y + np.random.uniform(-.08, .08, len(y)), FINAL['nest'], s=8, alpha=.5); ax[1].plot([1, 5], [1, 5], 'k--', lw=.8)
ax[1].set_xlabel('true score'); ax[1].set_ylabel('out-of-fold prediction'); ax[1].set_title(f"OOF predictions (RMSE {FINAL['cv_rmse']:.3f})")
res = pd.DataFrame({'label': y, 'err': FINAL['nest'] - y})
res.boxplot(column='err', by='label', ax=ax[2]); ax[2].axhline(0, color='k', lw=.8); ax[2].set_title('OOF error by true score'); ax[2].set_xlabel('true score'); plt.suptitle('')
plt.tight_layout(); plt.show()
print('per-score RMSE:', res.groupby('label').err.apply(lambda e: np.sqrt((e ** 2).mean())).round(3).to_dict())

A few notes on the plots. Text members get about a third of the weight even though every text member alone is weaker
than every audio member — their errors are only moderately correlated with the audio errors, which is exactly what
stacking needs. Errors are largest at the extremes (true 2 and true 5): a regressor trained on a mid-heavy label
distribution shrinks toward the middle, and the calibration slope above 1 only partly undoes it.

### The fine-tuned models
The Whisper-large-v3 encoder with LoRA (rank 16 on the top 16 layers, 5.9 M trainable parameters, 20-second crops,
the same speaker folds, 2.7 GPU hours) scores 0.578 alone and is in the stack with about 9 % weight. The earlier
WavLM-large full fine-tune was trained with random folds, so its out-of-fold predictions aren't clean enough to stack;
I blend it at a fixed 20 %, which I checked on the leaderboard (0.3537 → 0.3522).

I tried twice more to get more out of fine-tuning, and both attempts are worth recording because of *how* they failed.
Running the same LoRA recipe for 7 epochs instead of 4 (three folds were still improving at epoch 4) ended at 0.581 —
no better, and the validation curve swings by ±0.1 between epochs, so "best epoch" is mostly luck on 150 validation
clips. The second attempt was a distilled version: targets of 0.5·label + 0.5·(stack out-of-fold prediction), plus the
216 test clips as soft pseudo-labels so the encoder sees the test voices. Alone it scored 0.530, far better than any
other fine-tune, and as a stack member it took 57 % of the weight and pulled the nested CV from 0.512 to 0.505. The
leaderboard did not move at all (0.3485 → 0.3486). I think I know why: the teacher's out-of-fold prediction for a clip
in fold A was made by a model that had seen the labels of fold B, so when the student is trained on fold A and scored
on fold B, a little of fold B's label information has leaked in through its targets. The CV gain was the leak. I left
the distilled model out of the final stack. Fine-tuning was the most expensive part of the work and the smallest
contributor.

In [ ]:
ft = pd.read_csv(find('test_ft.csv')); assert (ft.filename.values == test.filename.values).all()
SPREAD = 1.1                                                   # predictions are shrunk toward the mean; 1.1 picked on the public LB (next section)
pred = 0.8 * FINAL['test'] + 0.2 * ft.pred.values
pred = np.clip(pred.mean() + SPREAD * (pred - pred.mean()), 1, 5)
pred[test.noise.values] = 0.0                                   # safety net; no test clip triggers it
sub = pd.DataFrame({'filename': test.filename, 'label': pred})
assert len(sub) == 216 and sub.label.notna().all()
sub.to_csv('submission.csv', index=False)
fig, ax = plt.subplots(figsize=(6, 3)); ax.hist(y, bins=np.arange(.75, 5.5, .5), alpha=.5, density=True, label='train labels')
ax.hist(pred, bins=20, alpha=.6, density=True, label='test predictions'); ax.legend(); ax.set_title('Prediction distribution vs training labels'); plt.show()
sub.describe().round(3).T

### Label noise, and two choices I made on the leaderboard
Two clips from the same speaker have labels that differ by about 0.47 (std). If a speaker really has one grammar
level, that means roughly 0.33 RMSE of rater noise per training clip — a floor no model can get under on these labels.
The leaderboard leader is at 0.306, so the test labels must be cleaner (averaged over raters, most likely).

That has two consequences that cross-validation cannot see, because CV is scored against the noisy labels:
1. Fitting the members to the mean label of each speaker's twin cluster (computed inside the training fold) makes CV
   slightly worse (0.508 → 0.512) but the leaderboard better (0.3515 → 0.3503).
2. Models trained on noisy labels shrink toward the mean; on cleaner labels it pays to undo some of that. Scaling the
   final prediction's distance from its mean by 1.1 improved the board (0.3515 → 0.3499); 0.9 hurt, 1.2 and above hurt.
   Together the two changes give 0.3485.

These are the only two decisions I took on the public leaderboard. Both are single smooth parameters with a clear
optimum, which limits how much I can be fitting the public subset; my second final submission is the untuned version
as a hedge.

Since the low scores are where the model is weakest (true-2.0 clips are predicted about 0.6 too high), I also tried
stretching only the low side of the predictions, and adding a "is this a 2 or below" classifier into the calibration.
Neither helped: the asymmetric stretch was worse on the board in all three settings I tried, and the classifier got a
coefficient near zero. The low end isn't a calibration problem — the features don't separate those clips.

In [ ]:
tw = nn_tr > 0.8; ds = y - y[j]                                   # signed label difference to the nearest same-voice clip
print(f'{tw.sum()} training clips have a same-voice twin (cos > 0.8): label difference std {ds[tw].std():.3f} '
      f'-> per-clip rater noise ~{ds[tw].std() / np.sqrt(2):.3f} RMSE (if a speaker has one true score)')
pd.DataFrame({'variant': ['reference', 'spread x0.9', 'spread x1.1', 'spread x1.2', 'denoised targets', 'denoised + x1.1', 'denoised + x1.15'],
              'grouped CV': [0.5079, '-', '-', '-', 0.5123, '-', '-'],
              'public LB': [0.3515, 0.3595, 0.3499, 0.3543, 0.3503, 0.3485, 0.3498]})

## 7. What is the model reacting to?
A sanity check with human-readable features from the transcripts: a zero-shot Qwen2.5-7B rubric grade, the
RoBERTa-CoLA acceptability probability, and speaking rate. They weren't needed in the stack (they add nothing on top
of the embeddings — see the next section), but they correlate with both the labels and my predictions, which is
reassuring: the model is tracking grammaticality, not loudness or clip length.

In [ ]:
judge = pd.read_csv(find('judge.csv')); assert (judge.filename.values == idx.filename.values).all()
F = pd.DataFrame({'LLM rubric grade (Qwen-7B, zero-shot)': judge.llm_score, 'LLM error rate /100 words': judge.llm_err_rate,
                  'CoLA acceptability (whole text)': judge.cola_whole, 'share of unacceptable sentences': judge.cola_bad,
                  'words per second': wt.n_words / np.r_[train.dur, test.dur], 'word count': wt.n_words, 'clip duration': np.r_[train.dur, test.dur]})
Ftr = F[KEEP].reset_index(drop=True)
corr = pd.DataFrame({'r with true score': [pearsonr(Ftr[c], y)[0] for c in F], 'r with our OOF prediction': [pearsonr(Ftr[c], FINAL['nest'])[0] for c in F]}, index=F.columns).round(3)
fig, ax = plt.subplots(figsize=(7, 3.2)); corr.plot.barh(ax=ax); ax.axvline(0, color='k', lw=.8); ax.set_title('Interpretable features vs score'); plt.tight_layout(); plt.show()
corr

In [ ]:
order = np.argsort(pred); ctc_te = ct.ctc.fillna('').values[IS_TE]
print('LOWEST predicted test clips (verbatim CTC transcript):')
for i in order[:2]:
    print(f'  {test.filename[i]} -> {pred[i]:.2f} | {ctc_te[i][:200]}')
print('\nHIGHEST predicted test clips:')
for i in order[-2:]:
    print(f'  {test.filename[i]} -> {pred[i]:.2f} | {ctc_te[i][:200]}')

## 8. What worked, what didn't, limitations
| step | grouped CV | public LB |
|---|---|---|
| constant (train mean) | – | 1.0050 |
| WavLM-L + Whisper-encoder layers, ridge/SVR, random folds | 0.484 (random) | 0.3862 |
| speaker-grouped CV + text track (RoBERTa-L, Qwen-1.5B) | 0.5219 | 0.3616 |
| + test-time twin smoothing | 0.5157 | 0.3604 |
| + Qwen-7B on the Whisper and the verbatim CTC transcript | 0.5093 | 0.3537 |
| + 20 % WavLM fine-tune blend | – | 0.3522 |
| + Whisper-encoder LoRA fine-tune as a stack member | 0.5073 | 0.3517 |
| + speaker-mean targets and spread 1.1 (this notebook) | 0.5123 | 0.3485 |

Tried under the same grouped folds and dropped because CV did not improve: Sarvam's Shuka/Saaras encoder (strong on
its own, redundant with WavLM/Whisper); the zero-shot LLM grade and CoLA as stack inputs (r ≈ 0.57 with the label but
≈ 0.07 with my residuals); 10-second segment-level training rows; speaker-averaged training rows; w2v-BERT 2.0; a
disfluency-prompted Whisper transcript (worse than CTC for the 7B reader); last-token "prompt" pooling of Qwen-7B;
a non-linear (gradient-boosting) stacker; Qwen2-Audio-7B hidden states and its zero-shot grade (the grade is nearly
constant at 3.6; the hidden states are redundant with the stack); regressing the speaker direction out of the audio
features before fitting (this removed most of the grammar signal too — on this data the voice and the grammar level
aren't separable linearly, which fits the fact that the errors are per-speaker); six fold seeds instead of three
(predictions correlate 0.9999 with the three-seed version); and the two extra fine-tunes described in section 6.

The pattern across all of it: every gain came from understanding the data (the noise clips, the repeated speakers, the
label noise, the transcript that keeps the errors), and nothing came from adding models. After the first week I had
twelve consecutive experiments that changed nothing, which is when I stopped.

Limitations. The public leaderboard is scored on a subset of 216 clips — I don't know the split, but at anything like
half of them the standard error of an RMSE around 0.35 is roughly 0.02. That means my last dozen submissions are
statistically indistinguishable, and so are the twenty teams on either side of me. With 100 submissions allowed per
day, scores well below that band on the public board are hard to tell from overfitting it; I made two decisions on the
board and kept an untuned hedge. The labels are single-rater MOS in 0.5 steps with roughly 0.33 RMSE of noise, so CV
ranks models reliably but doesn't predict the absolute test score. Everything is built on English-centric models;
accented or very quiet speech is where the two transcripts disagree most, and the true-2.0 clips are still where the
model is weakest (it scores them about 0.6 too high on average). The one thing I'd want for a real improvement is more
labelled data, ideally with two raters per clip — the rules of this competition don't allow external data, and that is
the actual bottleneck.

A few conventions that kept this manageable: inputs are found by glob because Kaggle's mount paths change; every
`.npy` shares one row order (`index.csv`); the speaker clustering is deterministic so the folds are identical
everywhere; and nothing uses a test label — the only test-side information is the test clips' own audio.